In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. 加载开源语义模型（把文本转成向量，不用自己造随机向量了）
model = SentenceTransformer('all-MiniLM-L6-v2')

/home/leovin/anaconda3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3795.88it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 2. 准备测试数据（模拟你的本地笔记/文档）
docs = []
docs.append("Faiss是Meta开源的向量检索库，支持多种索引类型，用于高维向量的快速相似性检索",)
docs.append("IVF索引通过k-means分桶实现检索加速，需要先训练聚类中心",)
docs.append("PQ乘积量化通过向量分段压缩，大幅减少内存占用，适合超大规模向量库",)
docs.append("HNSW是基于分层图的索引，检索速度极快，召回率接近暴力检索，适合线上实时场景",)
docs.append("Flat索引是暴力检索，召回率100%，适合小数据量场景",)
docs.append("Python是一门解释型编程语言，广泛用于机器学习、数据分析领域",)
docs.append("向量数据库的核心是向量检索引擎，很多底层基于Faiss实现",)
docs.append("RAG检索增强生成的核心步骤是：文档向量化、向量检索、prompt拼接、大模型生成",)

In [3]:
# 3. 把文本转成384维向量（对应我们之前学的dim参数）
dim = 384
doc_vectors = model.encode(docs).astype("float32")

In [4]:
# 4. 使用HNSW构建索引
index = faiss.index_factory(dim, "HNSW32", faiss.METRIC_L2)

# 因为HNSW的特性，不需要训练，直接添加向量即可
index.add(doc_vectors)
print(f"索引构建完成，向量总数：{index.ntotal}")

索引构建完成，向量总数：8


In [6]:
# 5. 执行索引（需要真正用Faiss了）
query = "Faiss的索引类型有哪些，分别有什么特点？"

# 因为是向量检索，所以需要把查询文本也转成向量
query_vector = model.encode([query]).astype("float32")

# 检索Top3最相似的文本
k = 3
D, I = index.search(query_vector, k)

In [8]:
# 6. 打印查询结果
print(f"我们的查询是: {query}")
print(f"在索引中检索到的最相关的内容为（top {k}）:")
for idx, (distance, doc_id) in enumerate(zip(D[0], I[0])):
    print(f"排名 {idx+1}: 距离={distance:.4f}, 文档ID={doc_id}, 内容={docs[doc_id]}")

我们的查询是: Faiss的索引类型有哪些，分别有什么特点？
在索引中检索到的最相关的内容为（top 3）:
排名 1: 距离=0.7001, 文档ID=6, 内容=向量数据库的核心是向量检索引擎，很多底层基于Faiss实现
排名 2: 距离=0.8114, 文档ID=0, 内容=Faiss是Meta开源的向量检索库，支持多种索引类型，用于高维向量的快速相似性检索
排名 3: 距离=0.9050, 文档ID=2, 内容=PQ乘积量化通过向量分段压缩，大幅减少内存占用，适合超大规模向量库
